In [1]:
import os
import pandas as pd
 
folder = os.getcwd()
products = pd.read_csv(folder + "\\Data\\amazon_products.csv")
categories = pd.read_csv(folder + "\\Data\\amazon_categories.csv")

In [2]:
products.columns = products.columns.str.strip()
products=products.drop(columns=['imgUrl','productURL'])
print("Columns:")
print(products.columns.tolist())
products=products[products['price']!=0]
print("\nNumber of products:", len(products))

Columns:
['Unnamed: 0', 'asin', 'title', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']

Number of products: 1393565


In [3]:
requirements = {
  "category_id": 123,
  "max_price": 20000,
  "min_stars": 4.0,
  "min_reviews": 100,
  "best_seller": True,
  "min_bought_last_month": 100
}

In [4]:
requirements = {
    "budget": 5000,
    "min_rating": 4.0,
    "min_reviews": 100,
    "items": [
        {"name": "item1","category_id": 56},
        {"name": "item2","category_id": 72}
    ]
}

In [5]:
domains = {}

for item in requirements["items"]:

    item_name = item["name"]
    category_id = item["category_id"]

    domain = products[
        (products["category_id"] == category_id) &
        (products["price"].notna()) &
        (products["stars"].notna()) &
        (products["reviews"].notna()) &
        (products["stars"] >= requirements["min_rating"]) &
        (products["reviews"] >= requirements["min_reviews"])
    ]

    domains[item_name] = domain

    print(
        item_name,
        "domain size:",
        len(domain)
    )

item1 domain size: 732
item2 domain size: 4731


In [6]:
def is_consistent(assignment, new_product, requirements):

    for product in assignment.values():

        if product["asin"] == new_product["asin"]:
            return False

    total_cost = new_product["price"]
    for product in assignment.values():
        total_cost += product["price"]

    if total_cost > requirements["budget"]:
        return False

    if new_product["stars"] < requirements["min_rating"]:
        return False
    if new_product["reviews"] < requirements["min_reviews"]:
        return False

    return True

In [7]:
nodes_explored = 0
backtracks = 0


def backtracking_search(variables,domains,assignment,requirements):
    global nodes_explored
    global backtracks

    if len(assignment) == len(variables):

        return assignment.copy()

    variable = variables[len(assignment)]
    
    for index in domains[variable].index:
        nodes_explored += 1
        product = products.loc[index]
        if is_consistent(assignment,product,requirements):
            assignment[variable] = product
            result = backtracking_search(variables,domains,assignment,requirements)

            if result is not None:
                return result

            del assignment[variable]
            backtracks += 1

    return None

In [8]:
variables = []
for item in requirements["items"]:
  variables.append(item["name"])

nodes_explored = 0
backtracks = 0

solution = backtracking_search(variables,domains,{},requirements)

if solution is None:
  print("NO FEASIBLE SOLUTION FOUND")
else:
  print("FEASIBLE SOLUTION FOUND\n")
  total_cost = 0
  for variable in variables:
    product = solution[variable]
    print("Variable:", variable)
    print("Product:", product["title"])
    print("Price:", product["price"])
    print("Rating:", product["stars"])
    print("Reviews:", product["reviews"])
    print("ASIN:", product["asin"])
    print()
    total_cost += product["price"]

  print("-------------------------")
  print("Total cost:", total_cost)
  print("Budget:", requirements["budget"])
  print("Nodes explored:", nodes_explored)
  print("Backtracks:", backtracks)

FEASIBLE SOLUTION FOUND

Variable: item1
Product: Sceptre 22 inch 75Hz 1080P LED Monitor 99% sRGB HDMI X2 VGA Build-In Speakers, Machine Black (E225W-19203R series)
Price: 68.97
Rating: 4.5
Reviews: 8549
ASIN: B07774L6TT

Variable: item2
Product: ScanSnap iX1600 Wireless or USB High-Speed Cloud Enabled Document, Photo & Receipt Scanner with Large Touchscreen and Auto Document Feeder for Mac or PC, Black
Price: 419.99
Rating: 4.7
Reviews: 3495
ASIN: B08PH5Q51P

-------------------------
Total cost: 488.96000000000004
Budget: 5000
Nodes explored: 2
Backtracks: 0
